# TNBike Sales — Data Cleaning & Linear Regression

**Adapted from WQU ADSL Module 1 (Housing in Mexico — Cleaning & Modeling)**

In this notebook we:
- Clean the sales dataset (handle missing values, outliers, type casting)
- Build a `wrangle()` function (WQU pattern)
- Establish a baseline model
- Train a Linear Regression: predict `total_amount` from `quantity + unit_price + discount`
- Evaluate with MAE and R²

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 2. Load Configuration

In [ ]:
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

db  = config['database']
prj = config['project']

TARGET   = prj['target']       # 'total_amount'
FEATURES = prj['features']     # ['quantity', 'unit_price', 'discount']

print(f'Target   : {TARGET}')
print(f'Features : {FEATURES}')

## 3. Connect to PostgreSQL

In [ ]:
engine = create_engine(
    f"postgresql://{db['user']}:{db['password']}@{db['host']}:{db['port']}/{db['dbname']}"
)

with engine.connect() as conn:
    ver = conn.execute(text('SELECT version()')).fetchone()[0]
print('DB:', ver[:50])

## 4. `wrangle()` Function — WQU Pattern

Encapsulate all loading + cleaning steps in a single reusable function.

In [ ]:
def wrangle(engine, start='2025-01-01', end='2026-03-31'):
    """
    Load TNBike fact_sales joined with dim_territory and dim_product.
    Perform cleaning steps and return a ready-to-model DataFrame.
    """
    query = f"""
        SELECT
            fs.order_id,
            fs.order_date,
            fs.quantity,
            fs.unit_price,
            fs.total_amount,
            fs.discount,
            dt.territory_name,
            dt.region,
            dp.category,
            dp.subcategory
        FROM tnbike.fact_sales fs
        LEFT JOIN tnbike.dim_territory dt ON fs.territory_id = dt.territory_id
        LEFT JOIN tnbike.dim_product   dp ON fs.product_id   = dp.product_id
        WHERE fs.order_date BETWEEN '{start}' AND '{end}'
    """
    df = pd.read_sql_query(query, engine)

    # --- Type casting ---
    df['order_date'] = pd.to_datetime(df['order_date'])
    for col in ['quantity', 'unit_price', 'total_amount', 'discount']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # --- Drop duplicates ---
    n_before = len(df)
    df = df.drop_duplicates(subset='order_id')
    print(f'Duplicates removed : {n_before - len(df)}')

    # --- Remove negative / zero values ---
    df = df[(df['quantity'] > 0) & (df['total_amount'] > 0) & (df['unit_price'] > 0)]

    # --- Clip discount to [0, 1] ---
    df['discount'] = df['discount'].clip(0, 1).fillna(0)

    # --- Outlier removal: IQR on total_amount ---
    q1, q3 = df['total_amount'].quantile([0.01, 0.99])
    df = df[(df['total_amount'] >= q1) & (df['total_amount'] <= q3)]

    # --- Fill remaining NaN ---
    df['region']         = df['region'].fillna('Unknown')
    df['territory_name'] = df['territory_name'].fillna('Unknown')
    df['category']       = df['category'].fillna('Unknown')
    df['subcategory']    = df['subcategory'].fillna('Unknown')

    print(f'Clean shape : {df.shape}')
    return df.reset_index(drop=True)


df = wrangle(engine)
df.head()

## 5. Post-Cleaning Inspection

In [ ]:
print('Shape  :', df.shape)
print()
print('Missing values:')
print(df.isnull().sum())
print()
print('Numeric summary:')
df[['quantity', 'unit_price', 'total_amount', 'discount']].describe()

## 6. Feature Engineering

In [ ]:
# Derived features
df['revenue_before_discount'] = df['quantity'] * df['unit_price']
df['discount_amount']         = df['revenue_before_discount'] * df['discount']
df['month']                   = df['order_date'].dt.month
df['quarter']                 = df['order_date'].dt.quarter
df['dayofweek']               = df['order_date'].dt.dayofweek

FEATURES_ENG = ['quantity', 'unit_price', 'discount',
                'revenue_before_discount', 'discount_amount',
                'month', 'quarter', 'dayofweek']

print('Engineered features added:')
df[FEATURES_ENG].describe()

## 7. Train / Test Split

In [ ]:
X = df[FEATURES_ENG]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train size : {len(X_train):,}')
print(f'Test  size : {len(X_test):,}')
print(f'Target mean (train): ${y_train.mean():,.2f}')

## 8. Baseline Model

WQU pattern: always establish a naive baseline before training a real model.

In [ ]:
# Baseline: predict the training-set mean for every observation
y_pred_baseline = np.full(len(y_test), y_train.mean())

baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
baseline_r2  = r2_score(y_test, y_pred_baseline)

print('=== Baseline (Mean Predictor) ===')
print(f'  MAE : {baseline_mae:,.4f}')
print(f'  R²  : {baseline_r2:.4f}')

## 9. Linear Regression Model

In [ ]:
# Impute any remaining NaN (safety)
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_imp, y_train)

y_pred_lr = lr.predict(X_test_imp)

lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_r2  = r2_score(y_test, y_pred_lr)

print('=== Linear Regression ===')
print(f'  MAE : {lr_mae:,.4f}')
print(f'  R²  : {lr_r2:.4f}')
print()
print(f'Improvement vs Baseline — MAE : {baseline_mae - lr_mae:,.4f}')

## 10. Coefficients Inspection

In [ ]:
coef_df = pd.DataFrame({
    'feature'    : FEATURES_ENG,
    'coefficient': lr.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print('Linear Regression Coefficients:')
print(coef_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['steelblue' if c > 0 else 'coral' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Linear Regression — Feature Coefficients', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('coefficients.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Residuals Analysis

In [ ]:
residuals = y_test.values - y_pred_lr

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_lr, alpha=0.3, s=10, color='steelblue')
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect')
axes[0].set_title('Actual vs Predicted', fontsize=12)
axes[0].set_xlabel('Actual total_amount ($)')
axes[0].set_ylabel('Predicted total_amount ($)')
axes[0].legend()

# Residuals distribution
sns.histplot(residuals, bins=50, kde=True, color='coral', ax=axes[1])
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Residuals Distribution', fontsize=12)
axes[1].set_xlabel('Residual ($)')

plt.tight_layout()
plt.savefig('residuals.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Residual mean : {residuals.mean():,.4f}')
print(f'Residual std  : {residuals.std():,.4f}')

## 12. Sales Cleaning Summary — Before vs After

In [ ]:
# Quick visual: total_amount distribution after cleaning
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df['total_amount'], bins=60, kde=True, color='steelblue', ax=ax)
ax.set_title('Cleaned total_amount Distribution', fontsize=13)
ax.set_xlabel('total_amount ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

print(f'Skewness after cleaning : {df["total_amount"].skew():.4f}')

## 13. Region-Level Prediction Analysis

In [ ]:
# Attach predictions to test set for analysis
test_df = X_test.copy()
test_df['actual']    = y_test.values
test_df['predicted'] = y_pred_lr
test_df['region']    = df.loc[X_test.index, 'region'].values

region_perf = test_df.groupby('region').apply(
    lambda g: pd.Series({
        'mae' : mean_absolute_error(g['actual'], g['predicted']),
        'r2'  : r2_score(g['actual'], g['predicted']) if len(g) > 1 else np.nan,
        'n'   : len(g)
    })
).reset_index()

print('Model Performance by Region:')
print(region_perf.to_string(index=False))

## 14. Monthly Revenue Trend (Cleaned Data)

In [ ]:
monthly = df.resample('M', on='order_date')['total_amount'].sum().reset_index()
monthly.columns = ['month', 'revenue']

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly['month'], monthly['revenue'], marker='o', color='steelblue', linewidth=2)
ax.fill_between(monthly['month'], monthly['revenue'], alpha=0.15, color='steelblue')
ax.set_title('Monthly Revenue — Cleaned Dataset', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=120, bbox_inches='tight')
plt.show()

## 15. Final Results & Communicate

In [ ]:
print('=' * 55)
print('  PROJECT 1 — RESULTS SUMMARY')
print('  TNBike Sales by Territory — Linear Regression')
print('=' * 55)
print(f'  Dataset rows (clean)   : {len(df):,}')
print(f'  Train / Test split     : 80% / 20%')
print()
print('  BASELINE (Mean Predictor)')
print(f'    MAE : ${baseline_mae:,.2f}')
print(f'    R²  : {baseline_r2:.4f}')
print()
print('  LINEAR REGRESSION')
print(f'    MAE : ${lr_mae:,.2f}')
print(f'    R²  : {lr_r2:.4f}')
print()
improvement_pct = (baseline_mae - lr_mae) / baseline_mae * 100
print(f'  MAE improvement vs baseline : {improvement_pct:.1f}%')
print()
print('  Top feature by coefficient magnitude:')
print(f'    {coef_df.iloc[0]["feature"]} ({coef_df.iloc[0]["coefficient"]:+.4f})')
print('=' * 55)
print('Proceed to Project 2: 2_housing_buenos_aires/')